In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [ ]:
def make_simple_model(in_ch: int, out_ch: int) -> nn.Sequential:
    """
    Простая модель: Linear -> ReLU -> Linear (без bias) -> ReLU
    реализована через nn.Sequential.
    """
    model = nn.Sequential(
        nn.Linear(in_ch, 32),  # умножает входной вектор размера in_ch на матрицу весов и добавляет смещение (bias).
        nn.ReLU(),
        nn.Linear(32, out_ch, bias=False),
        nn.ReLU(),
    )
    return model

# пример
simple_model = make_simple_model(in_ch=10, out_ch=3)


In [4]:
class MLP256_4(nn.Module):
    """
    Модель прямого прохода:
    256 -> 64 -> 16 -> 4 с активациями ReLU, tanh, Softmax.
    """
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(256, 64)
        self.fc2 = nn.Linear(64, 16)
        self.fc3 = nn.Linear(16, 4)

        self.relu = nn.ReLU()
        self.tanh = nn.Tanh()
        self.softmax = nn.Softmax(dim=1)  # Softmax по размерности классов

    def forward(self, x):
        """
        x: тензор формы (batch_size, 256)
        """
        x = self.relu(self.fc1(x))   # 256 -> 64
        x = self.tanh(self.fc2(x))   # 64 -> 16
        x = self.fc3(x)              # 16 -> 4 (логиты)
        x = self.softmax(x)          # вероятности по 4 классам
        return x

# пример
mlp_model = MLP256_4()
dummy = torch.randn(5, 256)
out = mlp_model(dummy)
print(out.shape)   # torch.Size([5, 4])


torch.Size([5, 4])


In [5]:
class ConvBlock(nn.Module):
    """
    2 свёртки + ReLU + 2 MaxPool:
    (N, 3, 19, 19) -> (N, 16, 4, 4)
    """
    def __init__(self):
        super().__init__()
        # 19x19 -> 18x18, каналы: 3 -> 8
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=8,
                               kernel_size=2, stride=1, padding=0)
        # 18x18 -> 9x9
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)

        # 9x9 -> 8x8, каналы: 8 -> 16
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16,
                               kernel_size=2, stride=1, padding=0)
        # 8x8 -> 4x4
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)

        self.relu = nn.ReLU()

    def forward(self, x):
        """
        x: (batch_size, 3, 19, 19)
        """
        x = self.relu(self.conv1(x))   # -> (B, 8, 18, 18)
        x = self.pool1(x)              # -> (B, 8, 9, 9)

        x = self.relu(self.conv2(x))   # -> (B, 16, 8, 8)
        x = self.pool2(x)              # -> (B, 16, 4, 4)

        return x

# пример
conv_block = ConvBlock()
img = torch.randn(2, 3, 19, 19)
feat = conv_block(img)
print(feat.shape)   # torch.Size([2, 16, 4, 4])


torch.Size([2, 16, 4, 4])


In [6]:
class FullImageClassifier(nn.Module):
    """
    Объединённая модель:
    Изображение 19x19x3 -> ConvBlock -> flatten (256) -> MLP256_4 -> 4-мерный вектор.
    """
    def __init__(self):
        super().__init__()
        self.conv_block = ConvBlock()
        self.mlp = MLP256_4()

    def forward(self, x):
        """
        x: (batch_size, 3, 19, 19)
        """
        x = self.conv_block(x)               # (B, 16, 4, 4)
        x = x.view(x.size(0), -1)            # (B, 256)
        x = self.mlp(x)                      # (B, 4)
        return x

# пример
model = FullImageClassifier()
dummy_img = torch.randn(4, 3, 19, 19)
out = model(dummy_img)
print(out.shape)   # torch.Size([4, 4])


torch.Size([4, 4])
